In [1]:
import trimesh

trimesh.interfaces.blender.exists

True

In [2]:
import trimesh.interfaces.blender as blender
print("Blender available:", blender.exists)

Blender available: True


In [3]:
import trimesh
import numpy as np
from trimesh.creation import box
from trimesh.transformations import translation_matrix

def create_plane_cutters(mesh, nx=3, ny=3):
    bounds_min, bounds_max = mesh.bounds
    zmin, zmax = bounds_min[2], bounds_max[2]

    x_vals = np.linspace(bounds_min[0], bounds_max[0], nx + 1)
    y_vals = np.linspace(bounds_min[1], bounds_max[1], ny + 1)

    planes = []
    for i in range(nx):
        for j in range(ny):
            xmin, xmax = x_vals[i], x_vals[i + 1]
            ymin, ymax = y_vals[j], y_vals[j + 1]
            center = [(xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2]
            normal = np.array([0, 0, 1])  # or modify per plane orientation
            planes.append((center, normal))
    return planes

def create_halfspace_box(origin, normal, size=10000):
    """Create a huge box to represent one side of a plane."""
    center = np.array(origin) + np.array(normal) * (size / 2)
    box_mesh = box(extents=[size, size, size], transform=translation_matrix(center))
    return box_mesh

def cut_mesh_with_planes(mesh, planes, output_prefix="plane_cut"):
    pieces = []
    for i, (origin, normal) in enumerate(planes):
        try:
            cutter = create_halfspace_box(origin, normal)
            piece = mesh.intersection(cutter, engine='blender')
            if piece.is_empty:
                print(f"Piece {i} is empty.")
                continue
            piece.export(f"{output_prefix}_{i}.stl")
            pieces.append(piece)
        except Exception as e:
            print(f"Failed to cut piece {i}: {e}")
    return pieces

def split_mesh_by_planes(mesh, nx=3, ny=3):
    planes = create_plane_cutters(mesh, nx, ny)
    pieces = cut_mesh_with_planes(mesh, planes, output_prefix="plane_piece")
    return pieces

def view_meshes(meshes, title="Mesh Viewer"):
    scene = trimesh.Scene()
    for mesh in meshes:
        scene.add_geometry(mesh)
    return scene.show(viewer="jupyter")

def recombine_meshes(meshes):
    return trimesh.util.concatenate(meshes)


In [8]:
pip install meshcut --user

  Using cached meshcut-0.3.0-py3-none-any.whl.metadata (2.2 kB)
Using cached meshcut-0.3.0-py3-none-any.whl (7.0 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
input_file = "C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Map_files/Wiss/Wiss.stl"  # change this
#mesh = trimesh.load_mesh(input_file)


In [ ]:
import meshcut
import numpy as np
from stl import mesh

# Load the STL file
your_mesh = mesh.Mesh.from_file(input_file )

# Optionally, save the cut parts as new STL files
if mesh_parts:
    for i, part in enumerate(mesh_parts):
        new_mesh = mesh.Mesh(np.zeros(part.shape[0], dtype=mesh.Mesh.dtype))
        for j, f in enumerate(part):
            new_mesh.vectors[j] = your_mesh.vectors[f]
        new_mesh.save(f'cut_part_{i}.stl')

NameError: name 'mesh_parts' is not defined

In [5]:

pieces = split_mesh_by_planes(mesh, nx=2, ny=2)
#view_meshes(pieces)


Piece 0 is empty.
Piece 1 is empty.
Piece 2 is empty.
Piece 3 is empty.


In [ ]:
mesh

In [ ]:
pieces

In [ ]:

print("=== Mesh Summary ===")
print(mesh)

print("\n=== Basic Properties ===")
print(f"Vertices: {len(mesh.vertices)}")
print(f"Faces: {len(mesh.faces)}")
print(f"Is watertight: {mesh.is_watertight}")
print(f"Is manifold: {mesh.is_watertight and mesh.is_winding_consistent}")
print(f"Euler number: {mesh.euler_number}")
print(f"Volume: {mesh.volume:.3f}")
print(f"Surface area: {mesh.area:.3f}")
print(f"Bounds: min={mesh.bounds[0]}, max={mesh.bounds[1]}")

print("\n=== Degeneracies ===")
print(f"Degenerate faces: {mesh.faces_sparse.shape[0] - mesh.faces.shape[0]}")
print(f"Zero area faces: {np.sum(mesh.area_faces == 0)}")
print(f"Non-watertight edges: {len(mesh.edges_unique) - len(mesh.edges_sorted)}")

print("\n=== Normals ===")
print(f"Average normal: {mesh.face_normals.mean(axis=0)}")

In [ ]:

# Step 1: Split into grid
print("Splitting mesh into jigsaw pieces...")
pieces = cut_pieces(mesh,cutters, output_prefix="jigsaw_piece")


In [ ]:
origin = mesh.centroid
normal = [0, 0, 1]  # Slice horizontally

# Get faces on each side of the plane
sliced = mesh.slice_plane(plane_origin=origin, plane_normal=normal)

In [21]:
import trimesh
import numpy as np

def split_mesh_with_planes(mesh, nx=3, ny=3, z_margin=5, export=False, prefix="piece"):
	bounds_min, bounds_max = your_mesh.min_,your_mesh.max_
	zmin, zmax = bounds_min[2], bounds_max[2]
	
	# Create X and Y grid lines
	x_vals = np.linspace(bounds_min[0], bounds_max[0], nx + 1)
	y_vals = np.linspace(bounds_min[1], bounds_max[1], ny + 1)
	
	pieces = []
	
	for i in range(nx):
		for j in range(ny):
			# Define bounding box region via 4 cutting planes
			xmin, xmax = x_vals[i], x_vals[i + 1]
			ymin, ymax = y_vals[j], y_vals[j + 1]

			# Start with the full mesh
			sub = mesh
			
			# Define planes: each as (origin, normal)
			planes = [
				# Cut left of xmin
				(np.array([xmin, 0, 0]), np.array([1, 0, 0])),
				# Cut right of xmax
				(np.array([xmax, 0, 0]), np.array([-1, 0, 0])),
				# Cut below ymin
				(np.array([0, ymin, 0]), np.array([0, 1, 0])),
				# Cut above ymax
				(np.array([0, ymax, 0]), np.array([0, -1, 0]))
			]
			
			for origin, normal in planes:
				try:
					

					# Cut the mesh
					sub = meshcut.cross_section(mesh, plane_orig=origin, plane_normal=normal)

				except Exception as e:
					print(f"Slice failed at cell ({i},{j}): {e}")
					sub = None
					break
			
			if sub and not sub.is_empty:
				pieces.append(sub)
				if export:
					sub.export(f"{prefix}_{i}_{j}.stl")
	
	return pieces

In [ ]:
meshcut.cross_section

In [22]:
pieces = split_mesh_with_planes(your_mesh , nx=2, ny=2, export=True)

Slice failed at cell (0,0): cross_section() missing 1 required positional argument: 'tris'
Slice failed at cell (0,1): cross_section() missing 1 required positional argument: 'tris'
Slice failed at cell (1,0): cross_section() missing 1 required positional argument: 'tris'
Slice failed at cell (1,1): cross_section() missing 1 required positional argument: 'tris'


In [ ]:
combined = recombine_meshes(pieces)

# Export to STL
combined.export("combined_grid_output.stl")

In [ ]:
mesh.fill_holes()

[]

In [21]:
sub = pieces[0].fill_holes()

In [ ]:
# Use mesh bounds
bounds_min, bounds_max = mesh.bounds
center = (bounds_min + bounds_max) / 2
size = [200, 200, 200]  # Make sure it overlaps the thin Z region


In [ ]:
[p.bounds for p in cutters]

In [ ]:
mesh.invert()


In [ ]:

# Step 2: View pieces interactively
print("Displaying jigsaw pieces...")
view_meshes([mesh], title="Jigsaw Parts")


In [ ]:

# Step 3: Recombine and view
print("Recombining parts...")
combined = recombine_meshes(pieces)
combined.export("recombined.stl")
view_meshes([combined], title="Recombined Mesh")
